# Native CLM Architecture Lab — Phase B

This is the single long-lived Native CLM architecture-search notebook.

Phase A found C1 (`residual gated update`) as the first clear CLM quality win. Phase B now separates two questions:

- **M-series** — generic modern-LM technology transferable to CLM: RMSNorm, SwiGLU, RoPE, then their combination.
- **N-series** — CLM-native structure: gate ablation, dual-gate ARU, recurrent-step-conditioned writes, adaptive communication.

C0, T1 and C1 are immutable anchors. Phase B does not retrain them.


In [ ]:
BRANCH = "research/native-clm-lab"
PROFILE = "baseline"
SEED = 91001
TRACK = "all"                 # all | modernization | native
EXPLICIT_MODELS = None        # e.g. ["M1-c1-rmsnorm", "M2-c1-swiglu"]
RUN_PHASE_B = True
ALLOW_CPU = False
PUSH_DEV_RESULT = True

assert SEED == 91001, "Phase B must not consume held-back seeds"


## Environment bootstrap

On Kaggle, the notebook updates the existing `research/native-clm-lab` checkout before execution so a stale Phase-A clone cannot silently run old code. Large caches, optimizer state and resume checkpoints remain under `/kaggle/working/native-clm`.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

def run_checked(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

repo_candidates = [Path.cwd(), Path("/kaggle/working/mini-cells")]
REPO_ROOT = next((p for p in repo_candidates if (p / ".git").exists()), None)

if REPO_ROOT is None:
    if not Path("/kaggle/working").exists():
        raise RuntimeError("Run inside the repository or on Kaggle")
    REPO_ROOT = Path("/kaggle/working/mini-cells")
    run_checked([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/ArcheLabs/mini-cells.git", str(REPO_ROOT),
    ])
elif Path("/kaggle/working").exists():
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_ROOT)
    run_checked(["git", "switch", BRANCH], cwd=REPO_ROOT)
    run_checked(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_ROOT)

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT) + "[lm]"])

WORK_ROOT = Path("/kaggle/working/native-clm") if Path("/kaggle/working").exists() else REPO_ROOT / ".native-clm-work"
CACHE_ROOT = WORK_ROOT / "cache"
OUTPUT_ROOT = WORK_ROOT / "runs"
HF_ROOT = WORK_ROOT / "huggingface"
for path in (CACHE_ROOT, OUTPUT_ROOT, HF_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(HF_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_ROOT / "datasets"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(HF_ROOT / "hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("repo:", REPO_ROOT)
print("head:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip())
print("work:", WORK_ROOT)


In [ ]:
def read_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = read_secret("HF_TOKEN")
GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print({
    "HF_TOKEN_available": bool(HF_TOKEN),
    "GITHUB_TOKEN_available": bool(GITHUB_TOKEN),
})


## Phase-B protocol audit

Every candidate must remain within 1% of the frozen T1 parameter count. A tiny forward smoke is also run before consuming accelerator budget.

The default eight-candidate schedule on two GPUs is:

```text
M1 / M2
M3 / M4
N1 / N2
N3 / N4
```


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "research/native-clm"))

import torch
import native_clm_runtime as base
from native_clm_phase_b import (
    M_NAMES, N_NAMES, PHASE_B_NAMES,
    build_phase_b, estimate_flops, parameter_summary, phase_b_config,
)

if EXPLICIT_MODELS is not None:
    SELECTED_MODELS = list(EXPLICIT_MODELS)
elif TRACK == "modernization":
    SELECTED_MODELS = list(M_NAMES)
elif TRACK == "native":
    SELECTED_MODELS = list(N_NAMES)
elif TRACK == "all":
    SELECTED_MODELS = list(PHASE_B_NAMES)
else:
    raise ValueError(TRACK)

params = parameter_summary()
for name in SELECTED_MODELS:
    assert params[name]["relative_error"] < 0.01, (name, params[name])

smoke_ids = torch.randint(0, base.VOCAB_SIZE, (1, 8))
for name in SELECTED_MODELS:
    model = build_phase_b(name).eval()
    with torch.no_grad():
        logits = model(smoke_ids)
    assert logits.shape == (1, 8, base.VOCAB_SIZE)
    assert torch.isfinite(logits).all()
    del model

audit = {
    "profile": PROFILE,
    "seed": SEED,
    "track": TRACK,
    "models": SELECTED_MODELS,
    "candidate_parameters": {name: params[name] for name in SELECTED_MODELS},
    "candidate_configs": {name: phase_b_config(name) for name in SELECTED_MODELS},
    "estimated_train_flops": {
        name: estimate_flops(name, base.PROFILES[PROFILE].target_tokens)["train_flops_estimate"]
        for name in SELECTED_MODELS
    },
}
print(json.dumps(audit, indent=2))


## Run Phase B

The runner reuses the frozen `train_one()` implementation. Optimizer, LR schedule, batch RNG, validation samples, checkpoint/resume and exact token accounting therefore remain the same as in C0/C1.

`N4` is a **soft** communication experiment: attention is still computed, so it carries no realized compute-saving claim.


In [ ]:
RUNNER = REPO_ROOT / "research/native-clm/run_native_clm_phase_b.py"
cmd = [
    sys.executable, str(RUNNER), "sweep",
    "--track", TRACK,
    "--profile", PROFILE,
    "--seed", str(SEED),
    "--cache-root", str(CACHE_ROOT),
    "--output-root", str(OUTPUT_ROOT),
]
if EXPLICIT_MODELS is not None:
    cmd += ["--models", *SELECTED_MODELS]
if ALLOW_CPU:
    cmd.append("--allow-cpu")

print("launching Phase B:", SELECTED_MODELS)
if RUN_PHASE_B:
    run_checked(cmd, cwd=REPO_ROOT)
else:
    print("RUN_PHASE_B=False; existing outputs only")


## Phase-B leaderboard

The leaderboard contains frozen T1/C0/C1 anchors plus the selected Phase-B candidates.

Primary quality columns are:

- `validation_ppl_10m`
- `log_token_nll_auc_mean`
- `ppl_over_c1`
- `ppl_over_t1`

Compute, throughput and VRAM remain separate evidence.


In [ ]:
RUN_DIR = OUTPUT_ROOT / f"phase-b-{PROFILE}-seed-{SEED}"
LEADERBOARD = RUN_DIR / "phase-b-leaderboard.json"
if not LEADERBOARD.is_file():
    raise FileNotFoundError(LEADERBOARD)

rows = json.loads(LEADERBOARD.read_text(encoding="utf-8"))
try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values("validation_ppl_10m"))
except Exception:
    print(json.dumps(sorted(rows, key=lambda row: row["validation_ppl_10m"]), indent=2))


## Record development evidence

Only small protocol, summary and learning-curve files are copied to Git. Checkpoints, optimizer state, token streams and caches remain outside the repository.


In [ ]:
record_dir = REPO_ROOT / "research/native-clm/results/dev" / f"phase-b-{PROFILE}-seed-{SEED}"
record_dir.mkdir(parents=True, exist_ok=True)

for name in ("protocol.json", "phase-b-leaderboard.json", "phase-b-leaderboard.csv", "phase-b-summary.json"):
    source = RUN_DIR / name
    if source.is_file():
        shutil.copy2(source, record_dir / name)

for model in SELECTED_MODELS:
    short = model.split("-")[0]
    for source_name, destination_name in (
        ("summary.json", f"{short}-summary.json"),
        ("checkpoints.csv", f"{short}-checkpoints.csv"),
    ):
        source = RUN_DIR / model / source_name
        if source.is_file():
            shutil.copy2(source, record_dir / destination_name)

ranked = sorted(rows, key=lambda row: row["validation_ppl_10m"])
lines = [
    "# Native CLM Phase-B development evidence",
    "",
    f"- profile: `{PROFILE}`",
    f"- seed: `{SEED}`",
    f"- track request: `{TRACK}`",
    "- status: development evidence; not NCLM-001",
    "",
    "## Leaderboard",
    "",
]
for row in ranked:
    lines.append(
        f"- {row['id']} ({row['track']}): "
        f"PPL={row['validation_ppl_10m']:.6f}, "
        f"AUC-mean={row['log_token_nll_auc_mean']:.6f}, "
        f"vs-C1={row['ppl_over_c1']:.6f}, params={row['parameters']}"
    )
(record_dir / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("development evidence:", record_dir)


In [ ]:
def sanitized_git_push(repo, record_dir, token):
    relative = record_dir.relative_to(repo)
    run_checked(["git", "config", "user.name", "native-clm-kaggle"], cwd=repo)
    run_checked(["git", "config", "user.email", "native-clm@users.noreply.github.com"], cwd=repo)
    run_checked(["git", "add", str(relative)], cwd=repo)
    changed = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode != 0
    if not changed:
        print("no new Phase-B evidence to push")
        return

    run_checked(["git", "commit", "-m", f"research: record Native CLM Phase-B seed {SEED}"], cwd=repo)
    if not token:
        print("GITHUB_TOKEN unavailable; committed locally only")
        return

    credential = (
        "protocol=https\n"
        "host=github.com\n"
        "username=x-access-token\n"
        f"password={token}\n\n"
    )
    subprocess.run(
        ["git", "credential", "approve"],
        input=credential,
        text=True,
        check=True,
        cwd=repo,
    )
    run_checked(["git", "push", "origin", f"HEAD:{BRANCH}"], cwd=repo)
    print("pushed Phase-B evidence to", BRANCH)

if PUSH_DEV_RESULT:
    sanitized_git_push(REPO_ROOT, record_dir, GITHUB_TOKEN)
else:
    print("automatic GitHub push disabled")
